In [1]:
!pip install transformers datasets peft trl accelerate bitsandbytes sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.3 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()  # select your Mental_Health_FAQ.csv when prompted

Saving Mental_Health_FAQ.csv to Mental_Health_FAQ.csv


In [4]:
import pandas as pd

df = pd.read_csv("Mental_Health_FAQ.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(98, 3)
['Question_ID', 'Questions', 'Answers']
   Question_ID                                    Questions  \
0      1590140  What does it mean to have a mental illness?   
1      2110618              Who does mental illness affect?   
2      6361820                  What causes mental illness?   

                                             Answers  
0  Mental illnesses are health conditions that di...  
1  It is estimated that mental illness affects 1 ...  
2  It is estimated that mental illness affects 1 ...  


In [6]:
!pip install huggingface_hub -q
from huggingface_hub import notebook_login
notebook_login()

In [7]:
# Format into prompt-response pairs
def format_row(row):
    return {
        "input_text": f"Answer this mental health question: {row['Questions']}",
        "target_text": row['Answers']
    }

formatted = df.apply(format_row, axis=1, result_type='expand')
print(formatted.head(2))

                                          input_text  \
0  Answer this mental health question: What does ...   
1  Answer this mental health question: Who does m...   

                                         target_text  
0  Mental illnesses are health conditions that di...  
1  It is estimated that mental illness affects 1 ...  


In [8]:
from datasets import Dataset

dataset = Dataset.from_pandas(formatted)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 88
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 10
    })
})


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded!


In [10]:
def tokenize(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        batch["target_text"],
        max_length=256,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = dataset.map(tokenize, batched=True)
print("Tokenization done!")

Map:   0%|          | 0/88 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenization done!


In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import os

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

args = TrainingArguments(
    output_dir="./flan-t5-mental-health",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,11.217776,9.940802
2,9.233735,7.529407
3,6.881003,5.029513
4,5.133300,3.440638
5,3.748918,3.333374
6,3.539405,3.260704
7,3.248074,3.201518
8,3.224643,3.151695
9,3.222242,3.119698
10,3.131709,3.108157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=110, training_loss=5.066322829506614, metrics={'train_runtime': 826.8359, 'train_samples_per_second': 1.064, 'train_steps_per_second': 0.133, 'total_flos': 150646617538560.0, 'train_loss': 5.066322829506614, 'epoch': 10.0})

In [15]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

test_question = "What causes mental illness?"
input_text = f"Answer this mental health question: {test_question}"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
outputs = model.generate(input_ids, max_length=200)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Q:", test_question)
print("A:", answer)

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausal

Q: What causes mental illness?
A: Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental illness. Mental illness is a disorder that affects the functioning of the brain, causing symptoms of mental ill

In [16]:
model.push_to_hub("mental-health-chatbot")
tokenizer.push_to_hub("mental-health-chatbot")
print("Pushed to Hugging Face!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p9ru1d3/model.safetensors:   0%|          |  554kB /  990MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed to Hugging Face!
